# Random forest and the bias-variance trade-off
One fully grown tree against a random forest, on a classification dataset (concentric circles) and a regression
dataset (two bumps plus noise). The figures of the Note are drawn by `images/figs.py` with the same data.

In [ ]:
import numpy as np
from sklearn.datasets import make_circles
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

## 1. Classification: concentric circles, 500 points

In [ ]:
# factor=0.1: the inner circle is small; noise=0.35: the two classes overlap
X, y = make_circles(n_samples=500, factor=0.1, noise=0.35, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

In [ ]:
# a fully grown tree: max_depth is left at None
tree = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
print("tree   train:", tree.score(X_train, y_train), " test:", tree.score(X_test, y_test))

# a random forest of 500 fully grown trees
rf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1).fit(X_train, y_train)
print("forest train:", rf.score(X_train, y_train), " test:", rf.score(X_test, y_test))

In [ ]:
# why the forest still scores 1.00 on training data:
# for each training point, what share of trees saw it, and do those trees get it right?
n = len(X_train)
inbag = np.zeros((len(rf.estimators_), n), bool)
correct = np.zeros_like(inbag)
for i, (t, rows) in enumerate(zip(rf.estimators_, rf.estimators_samples_)):
    inbag[i, np.unique(rows)] = True
    correct[i] = t.predict(X_train) == y_train
share = inbag.mean(axis=0)
print("share of trees that saw a point: min", share.min(), " mean", round(share.mean(), 3))
print("those trees are right on it:", correct[inbag].mean())
print("the other trees are right on it:", round(correct[~inbag].mean(), 3))

In [ ]:
# does more noise make the forest miss training points? change only the noise level
for noise in [0.35, 0.5, 0.8, 1.0]:
    Xn, yn = make_circles(n_samples=500, factor=0.1, noise=noise, random_state=42)
    Xn_train, _, yn_train, _ = train_test_split(Xn, yn, test_size=0.2, random_state=42)
    f = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1).fit(Xn_train, yn_train)
    print(f"noise {noise}: forest train accuracy {f.score(Xn_train, yn_train):.2f}")

## 2. Regression: two bumps plus noise
The same data as in the bagging regressor Note: 150 training points, here 1,000 test points.

In [ ]:
rng = np.random.RandomState(0)


def generate(n, noise=0.1):
    """x between -5 and 5; y = two Gaussian bumps (at 0 and at 2) plus noise."""
    x = np.sort(rng.rand(n) * 10 - 5)
    y = np.exp(-x ** 2) + 1.5 * np.exp(-(x - 2) ** 2) + rng.normal(0.0, noise, n)
    return x[:, None], y


X_train, y_train = generate(150)
X_test, y_test = generate(1000)

In [ ]:
dtr = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)
rfr = RandomForestRegressor(n_estimators=1000, random_state=0, n_jobs=-1).fit(X_train, y_train)
for name, model in [("tree", dtr), ("forest", rfr)]:
    pred = model.predict(X_test)
    print(f"{name:6s} train MSE {mean_squared_error(y_train, model.predict(X_train)):.4f}"
          f"  test MSE {mean_squared_error(y_test, pred):.4f}"
          f"  test sum of squared errors {np.sum((y_test - pred) ** 2):.1f}")